# 🌿 Crop Disease Detection using CNN (Transfer Learning)

**Dataset**: PlantVillage — 54,000+ leaf images across 38 disease classes (14 crops)  
**Model**: MobileNetV2 pretrained on ImageNet → fine-tuned for disease classification  
**Tools**: PyTorch, torchvision, scikit-learn, matplotlib

---
### Pipeline
```
Raw Images → Augmentation → MobileNetV2 backbone → Custom head → Train/Val/Test → Predict
```

## 1. Install Dependencies

In [1]:
# Run once if needed
# !pip install torch torchvision scikit-learn matplotlib seaborn pillow

## 2. Imports & Config

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix

CONFIG = {
    "data_dir": "./data/PlantVillage",  # <-- set your path
    "img_size": 224,
    "batch_size": 32,
    "epochs": 10,
    "lr": 1e-4,
    "val_split": 0.15,
    "test_split": 0.15,
    "num_workers": 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_path": "crop_disease_model.pth",
}

print(f"Device: {CONFIG['device']}")

KeyboardInterrupt: 

## 3. Dataset Setup

**Download PlantVillage from Kaggle:**
```bash
kaggle datasets download -d abdallahalidev/plantvillage-dataset
unzip plantvillage-dataset.zip -d ./data
```
Expected folder structure:
```
data/PlantVillage/
    Apple___Apple_scab/
    Apple___Black_rot/
    Corn___Common_rust/
    Tomato___Late_blight/
    ...
```

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load dataset
full_dataset = datasets.ImageFolder(root=CONFIG["data_dir"], transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Total images : {len(full_dataset)}")
print(f"Classes      : {num_classes}")
print(f"Sample classes: {class_names[:5]}")

In [ ]:
# Train / Val / Test split
total = len(full_dataset)
val_size  = int(total * CONFIG["val_split"])
test_size = int(total * CONFIG["test_split"])
train_size = total - val_size - test_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

val_ds.dataset.transform  = val_test_transform
test_ds.dataset.transform = val_test_transform

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"])
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print(f"Train: {train_size} | Val: {val_size} | Test: {test_size}")

## 4. Visualize Sample Images

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i]], fontsize=7)
    ax.axis('off')
plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Build Model — MobileNetV2 + Custom Head

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Linear(256, num_classes)
)
model = model.to(CONFIG["device"])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total_params:,}")

## 6. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG["lr"])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, CONFIG["epochs"] + 1):
    # --- Train ---
    model.train()
    t_loss, t_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(CONFIG["device"]), labels.to(CONFIG["device"])
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * images.size(0)
        t_correct += (out.argmax(1) == labels).sum().item()

    # --- Val ---
    model.eval()
    v_loss, v_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(CONFIG["device"]), labels.to(CONFIG["device"])
            out = model(images)
            v_loss += criterion(out, labels).item() * images.size(0)
            v_correct += (out.argmax(1) == labels).sum().item()

    scheduler.step()

    t_acc = t_correct / train_size
    v_acc = v_correct / val_size
    history["train_loss"].append(t_loss / train_size)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss / val_size)
    history["val_acc"].append(v_acc)

    print(f"Epoch [{epoch:02d}/{CONFIG['epochs']}]  "
          f"Train Acc: {t_acc:.4f}  Val Acc: {v_acc:.4f}")

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), CONFIG["save_path"])
        print(f"  ✓ Saved best model")

## 7. Training Curves

In [ ]:
epochs = range(1, CONFIG["epochs"] + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["val_loss"],   label="Val")
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], label="Train")
axes[1].plot(epochs, history["val_acc"],   label="Val")
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()

## 8. Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(CONFIG["save_path"], map_location=CONFIG["device"]))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        out = model(images.to(CONFIG["device"]))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(min(22, num_classes), min(18, num_classes)))
sns.heatmap(cm, annot=(num_classes <= 15), fmt="d",
            xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.title("Confusion Matrix — Test Set")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## 10. Predict on a Single Image

In [ ]:
from PIL import Image

def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = val_test_transform(img).unsqueeze(0).to(CONFIG["device"])

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
        top5  = probs.topk(5)

    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Top prediction: {class_names[top5.indices[0]]}\n"
              f"Confidence: {top5.values[0].item()*100:.1f}%")
    plt.tight_layout()
    plt.show()

    print("\nTop-5 Predictions:")
    for prob, idx in zip(top5.values, top5.indices):
        print(f"  {class_names[idx]:<40}  {prob.item()*100:.2f}%")

# Usage:
# predict("path/to/leaf_image.jpg")

---
## Summary

| Component | Details |
|---|---|
| Dataset | PlantVillage (~54K images, 38 classes) |
| Model | MobileNetV2 (pretrained ImageNet) |
| Strategy | Freeze backbone → fine-tune head |
| Augmentation | Flip, Rotation, ColorJitter |
| Optimizer | Adam (lr=1e-4) + StepLR scheduler |
| Expected accuracy | ~95%+ with GPU, ~88-92% CPU |

**To improve further:**
- Unfreeze last few backbone layers (fine-tuning)
- Try EfficientNet-B0 or ResNet50
- Use Grad-CAM for visualization
- Deploy with FastAPI + Streamlit